In [1]:
"""
Ridge, Lasso, and ElasticNet regression built from scratch with
(sub)gradient descent, plus a feature standardization helper.

Ridge  : L2 penalty  -> alpha * sum(w_j^2)
Lasso  : L1 penalty  -> alpha * sum(|w_j|)
ElasticNet: mix of both, controlled by l1_ratio in [0, 1]
    l1_ratio = 1.0 -> pure Lasso
    l1_ratio = 0.0 -> pure Ridge
"""

import numpy as np


def standardize(X):
    """Zero mean, unit variance per feature. Required before regularization
    so the penalty is applied fairly across features of different scales."""
    X = np.asarray(X, dtype=float)
    mean = X.mean(axis=0)
    std = X.std(axis=0)
    std = np.where(std == 0, 1.0, std)  # avoid divide-by-zero for constant columns
    return (X - mean) / std, mean, std


class RidgeRegression:
    def __init__(self, alpha=1.0, learning_rate=0.01, n_iters=1000):
        self.alpha = alpha
        self.learning_rate = learning_rate
        self.n_iters = n_iters
        self.weights = None
        self.bias = None

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0.0

        for _ in range(self.n_iters):
            y_pred = X @ self.weights + self.bias
            error = y_pred - y

            grad_w = (2 / n_samples) * (X.T @ error) + 2 * self.alpha * self.weights
            grad_b = (2 / n_samples) * np.sum(error)

            self.weights -= self.learning_rate * grad_w
            self.bias -= self.learning_rate * grad_b

        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        return X @ self.weights + self.bias


class LassoRegression:
    def __init__(self, alpha=1.0, learning_rate=0.01, n_iters=1000):
        self.alpha = alpha
        self.learning_rate = learning_rate
        self.n_iters = n_iters
        self.weights = None
        self.bias = None

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0.0

        for _ in range(self.n_iters):
            y_pred = X @ self.weights + self.bias
            error = y_pred - y

            grad_w = (2 / n_samples) * (X.T @ error) + self.alpha * np.sign(self.weights)
            grad_b = (2 / n_samples) * np.sum(error)

            self.weights -= self.learning_rate * grad_w
            self.bias -= self.learning_rate * grad_b

        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        return X @ self.weights + self.bias


class ElasticNetRegression:
    def __init__(self, alpha=1.0, l1_ratio=0.5, learning_rate=0.01, n_iters=1000):
        self.alpha = alpha
        self.l1_ratio = l1_ratio
        self.learning_rate = learning_rate
        self.n_iters = n_iters
        self.weights = None
        self.bias = None

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0.0

        for _ in range(self.n_iters):
            y_pred = X @ self.weights + self.bias
            error = y_pred - y

            l1_grad = self.l1_ratio * np.sign(self.weights)
            l2_grad = (1 - self.l1_ratio) * 2 * self.weights
            grad_w = (2 / n_samples) * (X.T @ error) + self.alpha * (l1_grad + l2_grad)
            grad_b = (2 / n_samples) * np.sum(error)

            self.weights -= self.learning_rate * grad_w
            self.bias -= self.learning_rate * grad_b

        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        return X @ self.weights + self.bias


def r_squared(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    return 1 - ss_res / ss_tot


if __name__ == "__main__":
    # Smoke test: 5 informative features + 15 pure-noise features
    rng = np.random.default_rng(42)
    n_samples, n_informative, n_noise = 200, 5, 15
    n_features = n_informative + n_noise

    X = rng.normal(size=(n_samples, n_features))
    true_w = np.array([3.0, -2.0, 1.5, 0.0, 4.0] + [0.0] * n_noise)
    y = X @ true_w + rng.normal(scale=1.0, size=n_samples)

    X_std, mean, std = standardize(X)

    ridge = RidgeRegression(alpha=1.0, learning_rate=0.1, n_iters=2000).fit(X_std, y)
    lasso = LassoRegression(alpha=0.5, learning_rate=0.1, n_iters=2000).fit(X_std, y)
    elastic = ElasticNetRegression(alpha=0.5, l1_ratio=0.5, learning_rate=0.1, n_iters=2000).fit(X_std, y)

    print("Ridge R^2:", round(r_squared(y, ridge.predict(X_std)), 4))
    print("Lasso R^2:", round(r_squared(y, lasso.predict(X_std)), 4))
    print("Lasso nonzero coefs:", int(np.sum(np.abs(lasso.weights) > 1e-3)), "/", n_features)
    print("ElasticNet R^2:", round(r_squared(y, elastic.predict(X_std)), 4))


Ridge R^2: 0.7473
Lasso R^2: 0.9641
Lasso nonzero coefs: 19 / 20
ElasticNet R^2: 0.9232
